In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset,WeightedRandomSampler
from torch.nn.utils.rnn import pad_sequence
from google.colab import drive
from collections import Counter
import torch.optim as optim
from sklearn.metrics import confusion_matrix,classification_report
import re
torch.manual_seed(42)

In [8]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

In [9]:

drive.mount('/content/drive')

Mounted at /content/drive


#DATA PROCESSING

In [10]:
test_df = pd.read_csv('/content/drive/MyDrive/Assignment_2_Dataset/Task 4/Data 4/test.csv')
train_df = pd.read_csv('/content/drive/MyDrive/Assignment_2_Dataset/Task 4/Data 4/train.csv')
GLOVE_PATH='/content/drive/MyDrive/Assignment_2_Dataset/Task 3/glove.6B.200d.txt'


In [11]:
test_df.head()

,Sentence,Tags
0,"`` We'll work hard Tuesday , Wednesday and Thu...",. PRT VERB ADV NOUN . NOUN CONJ NOUN . . NOUN ...
1,But they could turn and escape to the east .,CONJ PRON VERB VERB CONJ VERB ADP DET NOUN .
2,From next New Year's Day let us keep careful a...,ADP ADJ ADJ NOUN NOUN VERB PRON VERB ADJ NOUN ...
3,This condition will undoubtedly continue until...,DET NOUN VERB ADV VERB ADP ADJ NOUN ADP DET NO...
4,"In the course of this , they had fired on us ; ;",ADP DET NOUN ADP DET . PRON VERB VERB ADP PRON...


Basic Preprocess

In [12]:
Vocab = []
# Include both train and test tweets in vocabulary
for tweet in train_df['Sentence']:
    Vocab.extend(tweet.split())

Tags=[]
for tag in train_df['Tags']:
  Tags.extend(tag.split())


Word_to_inx

In [13]:
def word_idx_map(Vocab):
    word_counts = Counter(Vocab)
    # print(counts)
    word_to_idx = {"<PAD>": 0, "<UNK>": 1}
    for i, word in enumerate(word_counts):
        word_to_idx[word] = i + 2

    print(word_to_idx)
    vocab_size = len(word_to_idx) + 1
    return word_to_idx
    # print(f"vocab size: {vocab_size}")


In [14]:
word_to_idx=word_idx_map(Vocab) # {word:indx}
tag_to_idx=word_idx_map(Tags) #{tag:idx}



{'<PAD>': 0, '<UNK>': 1, 'If': 2, 'working': 3, 'in': 4, 'a': 5, 'zinc': 6, 'mine': 7, ',': 8, 'which': 9, 'he': 10, 'once': 11, 'did': 12, 'for': 13, '87-1/2': 14, 'cents': 15, 'an': 16, 'hour': 17, 'paid': 18, 'more': 19, 'than': 20, 'playing': 21, 'center': 22, 'field': 23, 'the': 24, 'Yankees': 25, 'Mantle': 26, 'would': 27, 'work': 28, '.': 29, 'There': 30, 'had': 31, 'not': 32, 'yet': 33, 'supervened': 34, 'between': 35, 'understanding': 36, 'and': 37, 'expression': 38, 'new': 39, 'languages': 40, 'of': 41, 'mathematics': 42, 'scientific': 43, 'formulas': 44, 'Less': 45, 'common': 46, 'seeds': 47, 'used': 48, 'cooking': 49, 'beverages': 50, 'include': 51, 'fenugreek': 52, '(': 53, 'artificial': 54, 'maple': 55, 'flavor': 56, ')': 57, 'cardamom': 58, '``': 59, 'God': 60, 'I': 61, 'take': 62, 'it': 63, 'plays': 64, 'no': 65, 'part': 66, 'this': 67, "''": 68, 'she': 69, 'said': 70, 'waspishly': 71, 'It': 72, 'landed': 73, 'on': 74, "Roberts'": 75, 'sprawled': 76, 'right': 77, 'thigh

In [15]:
def prepare_dataloader(df, wtoi,ttoi, batch_size=32, shuffle=False):
    """
    Converts a DataFrame of tokenized tweets into a PyTorch DataLoader.
    """
    X_list = []

    # 1. Convert words to IDs using the provided vocab
    # Assumes 'tweet' column contains a list of strings (tokens)
    for token in df['Sentence']:
        # Split the tweet into a list of words
        words = token.split()

        number_ids =[wtoi.get(word, 1) for word in words]
        X_list.append(torch.tensor(number_ids))

    # 2. Pad sequences to make them uniform
    X_padded = pad_sequence(X_list, batch_first=True, padding_value=0)
    Y_list=[]
    for token in df['Tags']:
        # Split the tweet into a list of words
        words = token.split()

        number_ids =[ttoi.get(word, 1) for word in words]
        Y_list.append(torch.tensor(number_ids))
    Y_padded = pad_sequence(Y_list, batch_first=True, padding_value=0)
    # 4. Create Dataset and DataLoader
    dataset = TensorDataset(X_padded,Y_padded)
    loader = DataLoader(dataset, batch_size=batch_size ,shuffle=shuffle)

    return loader

In [16]:
train_loader= prepare_dataloader(train_df, word_to_idx,tag_to_idx, batch_size=32, shuffle=True)
test_loader= prepare_dataloader(test_df, word_to_idx,tag_to_idx, batch_size=32, shuffle=False)


#Load the 200D GloVe Embeddings

In [17]:
def load_glove_embeddings(path, word_idx, embedding_dim=200):
    embeddings = np.zeros((len(word_idx) + 1, embedding_dim))
    with open(path, 'r', encoding='utf8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            if word in word_idx:
                idx = word_idx[word]
                embeddings[idx] = np.asarray(values[1:], dtype='float32')
    return torch.from_numpy(embeddings).float()

In [18]:
embedding_matrix = load_glove_embeddings(GLOVE_PATH, word_to_idx)

In [19]:
embedding_matrix.shape,len(word_to_idx)

(torch.Size([17889, 200]), 17888)

#Build the RNN Model

In [20]:
class POS_RNN(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim, tagset_size):
        super(POS_RNN, self).__init__()
        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=True)
        self.rnn = nn.RNN(input_size=200, hidden_size=hidden_dim, num_layers=1, batch_first=True)
        self.hidden2tag = nn.Linear(hidden_dim, tagset_size)

    def forward(self, sentences):
        embeds = self.embedding(sentences)
        rnn_out, _ = self.rnn(embeds)
        #rnn_out is the output of hidden layer at each time stamp returned after processing full sequnece
        tag_space = self.hidden2tag(rnn_out)
        # Return the 3D tensor (Batch, Seq, Classes) instead of flattening here
        return tag_space

In [21]:
VOCAB_SIZE = len(word_to_idx)
TAGSET_SIZE = len(tag_to_idx)
HIDDEN_DIM = 25

torch.manual_seed(42)
# Re-instantiate the model to ensure it uses the latest POS_RNN definition
model = POS_RNN(embedding_matrix, HIDDEN_DIM, TAGSET_SIZE)
model.to(device)
print(model)

POS_RNN(
  (embedding): Embedding(17889, 200)
  (rnn): RNN(200, 25, batch_first=True)
  (hidden2tag): Linear(in_features=25, out_features=14, bias=True)
)


In [22]:
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model.parameters(), lr=0.001)

#Compile and Train the Model

In [23]:
def train_model(model, train_loader, criterion, optimizer, epochs=5, pad_idx=0):
    model.to(device)
    print(f"Starting training on {device} for {epochs} epochs...")
    for epoch in range(epochs):
        model.train()
        total_loss, correct_predictions, total_valid_tokens = 0, 0, 0
        for X_batch, Y_batch in train_loader:
            X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)
            optimizer.zero_grad()
            # outputs shape: (Batch, Seq, Classes)
            outputs = model(X_batch)
            # Flatten for CrossEntropyLoss: (Batch*Seq, Classes)
            outputs_flattened = outputs.view(-1, outputs.shape[-1])
            targets_flattened = Y_batch.view(-1)
            loss = criterion(outputs_flattened, targets_flattened)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            _, predicted = torch.max(outputs_flattened, 1)
            mask = (targets_flattened != pad_idx)
            correct_predictions += (predicted[mask] == targets_flattened[mask]).sum().item()
            total_valid_tokens += mask.sum().item()
        avg_train_loss = total_loss / len(train_loader)
        train_accuracy = correct_predictions / total_valid_tokens if total_valid_tokens > 0 else 0
        print(f"Epoch [{epoch+1}/{epochs}] | Loss: {avg_train_loss:.4f} | Accuracy: {train_accuracy:.4f}")
    return model

In [24]:
optimizer = optim.Adam(model.parameters(), lr=0.001)

POS_Model = train_model(
    model=model,
    train_loader=train_loader,
    criterion=criterion,
    optimizer=optimizer,
    epochs=10,
    pad_idx=0
)

Starting training on cpu for 10 epochs...
Epoch [1/10] | Loss: 1.6655 | Accuracy: 0.5402
Epoch [2/10] | Loss: 0.9433 | Accuracy: 0.7194
Epoch [3/10] | Loss: 0.7480 | Accuracy: 0.7676
Epoch [4/10] | Loss: 0.6550 | Accuracy: 0.7962
Epoch [5/10] | Loss: 0.5989 | Accuracy: 0.8127
Epoch [6/10] | Loss: 0.5604 | Accuracy: 0.8236
Epoch [7/10] | Loss: 0.5318 | Accuracy: 0.8315
Epoch [8/10] | Loss: 0.5069 | Accuracy: 0.8395
Epoch [9/10] | Loss: 0.4865 | Accuracy: 0.8452
Epoch [10/10] | Loss: 0.4690 | Accuracy: 0.8509


#Evaluation and Reporting

In [25]:
def evaluate_test_data(model, test_loader, device, tag_to_idx, pad_idx=0):
    model.eval()
    all_preds, all_targets = [], []
    idx_to_tag = {v: k for k, v in tag_to_idx.items()}
    target_names = [idx_to_tag[i] for i in range(len(idx_to_tag)) if i != pad_idx]
    labels = [i for i in range(len(idx_to_tag)) if i != pad_idx]
    with torch.no_grad():
        for X_batch, Y_batch in test_loader:
            X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)
            outputs = model(X_batch)
            outputs_flattened = outputs.view(-1, outputs.shape[-1])
            targets_flattened = Y_batch.view(-1)
            _, predicted = torch.max(outputs_flattened, 1)
            mask = (targets_flattened != pad_idx)
            all_preds.extend(predicted[mask].cpu().numpy())
            all_targets.extend(targets_flattened[mask].cpu().numpy())
    accuracy = (np.array(all_preds) == np.array(all_targets)).mean()
    print(f"\n--- Final Test Evaluation ---")
    print(f"Test Accuracy (excluding padding): {accuracy:.4f}")
    print("\nClassification Report:")
    print(classification_report(all_targets, all_preds, labels=labels, target_names=target_names))
    return accuracy

In [26]:
evaluate_test_data(POS_Model, test_loader, device, tag_to_idx, pad_idx=0)


--- Final Test Evaluation ---
Test Accuracy (excluding padding): 0.8417

Classification Report:
              precision    recall  f1-score   support

       <UNK>       0.00      0.00      0.00         0
         ADP       0.91      0.86      0.88      4244
        VERB       0.88      0.80      0.84      5224
         DET       0.80      0.97      0.88      4038
        NOUN       0.75      0.91      0.82      7828
           .       1.00      1.00      1.00      4047
        PRON       0.89      0.71      0.79      1323
         ADV       0.78      0.63      0.70      1635
         NUM       0.86      0.73      0.79       393
         ADJ       0.78      0.48      0.59      2491
         PRT       0.73      0.71      0.72       846
        CONJ       0.99      0.91      0.95      1105
           X       0.00      0.00      0.00        32

    accuracy                           0.84     33206
   macro avg       0.72      0.67      0.69     33206
weighted avg       0.85      0.84    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/me

np.float64(0.841715352647112)